# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Inspect high-level metadata (name, description, etc.)
# Note: dataset.metadata is an object. Access fields with dot notation.
print(f"Dataset: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Citation: {dataset.metadata.citeAs}")
print(f"License: {dataset.metadata.license}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate the `@id`s of all available record sets, and for each, the fields and columns provided in the schema.

In [ ]:
# List all record sets
print('Available Record Sets:')
for rs in dataset.record_sets:
    print(f"- RecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")

    # List fields for this record set
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {getattr(field, 'data_type', None)})")
    # List columns for this record set
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {col.name} (@id: {col.id})")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. 

We'll use the `@id` of the record set(s) as shown above. For each, we extract a DataFrame and show the column names and first few rows.

In [ ]:
# Gather all record_set @ids
record_sets_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    records_iter = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(list(records_iter))
    dataframes[record_set_id] = df
    print(f"\nRecordSet {record_set_id} DataFrame:")
    print(df.columns.tolist())
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
Let's select one record set with numeric fields (e.g., protein quantitative metrics) and perform some exploratory analysis.

* Filter records based on a threshold
* Normalize a numeric field
* Group by another field and compute means

In [ ]:
# For demonstration, select the first RecordSet with a suitable numeric field
import numpy as np

# Helper: show all record sets and choose one for EDA
for i, rs in enumerate(dataset.record_sets):
    print(f"{i}: {rs.name}, @id: {rs.id}")

# Choose the record set of interest (update if known)
# Example: rs_idx = 0  # change if desired
rs_idx = 0
chosen_rs = dataset.record_sets[rs_idx]
chosen_rs_id = chosen_rs.id
df = dataframes[chosen_rs_id]
print(f"\nColumns for {chosen_rs_id}:", df.columns.tolist())

# Try to identify a numeric field automatically
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    numeric_field_id = numeric_cols[0]  # Take first numeric column
else:
    print("No numeric columns found in this record set.")
    numeric_field_id = None

if numeric_field_id:
    threshold = df[numeric_field_id].mean()  # Use mean as threshold example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by next non-numeric field for demonstration
    group_candidates = [c for c in df.columns if c != numeric_field_id]
    textual_group = None
    for c in group_candidates:
        if pd.api.types.is_string_dtype(df[c]) and df[c].nunique() < 20:
            textual_group = c
            break
    if textual_group:
        grouped_df = filtered_df.groupby(textual_group).mean(numeric_only=True)
        print(f"\nGrouped data by '{textual_group}':")
        display(grouped_df.head())
    else:
        print("No suitable text/group fields found for grouping.")
else:
    print("No numeric field selected for EDA.")


## 5. Visualization
Let's visualize the distribution of the chosen numeric field and, if a group field exists, average values per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Plot the histogram for the numeric field
if numeric_field_id and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} in record set {chosen_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Optional: boxplot by group
    if textual_group and textual_group in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=textual_group, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {textual_group}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")


## 6. Conclusion
In this notebook, we used the Croissant schema and the `mlcroissant` library to:

- Load and summarize dataset metadata
- List available record sets, fields, and columns by `@id`
- Extract tabular data for each record set
- Perform basic exploratory analysis and normalization on numeric fields, grouping by categorical fields where available
- Visualize the distribution of a selected numeric attribute

This process enables downstream statistical and machine learning analysis. For deeper exploration, please refer to the documentation for the `mlcroissant` library and to the [dataset's FAIR² landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa/).